# Detección de anomalías en las compras públicas del Ecuador mediante datos abiertos SERCOP

**Proyecto de tesis — Maestría en Ciencia de Datos e Inteligencia Artificial, UIDE**

## 1. Introducción

Este notebook documenta, paso a paso, el proceso de construcción de la base de datos y el análisis exploratorio del proyecto de tesis. El objetivo del proyecto es identificar procesos de contratación pública del Ecuador con patrones estadísticamente atípicos en precio, concurrencia de oferentes y tiempos de adjudicación, utilizando aprendizaje automático no supervisado.

**Catálogo UIDE, ítem #42**: "Detección de anomalías en grandes volúmenes de datos abiertos gubernamentales". Se diferencia del ítem #48 (ya tomado por otro grupo, enfocado en redes de colusión entre empresas) en que este proyecto trabaja detección de anomalías estadísticas/ML a nivel de **proceso individual**, no análisis de redes o grafos.

## 2. ¿De dónde vienen los datos?

La fuente es el portal de **Datos Abiertos de Contratación Pública del Ecuador (SERCOP)**:

- Portal: https://datosabiertos.compraspublicas.gob.ec/PLATAFORMA/datos-abiertos
- Estándar: **OCDS (Open Contracting Data Standard)** — un estándar internacional de datos abiertos de contratación pública, usado también en Chile, Colombia, México, Paraguay y Uruguay.
- Cobertura: procesos de contratación pública del Ecuador desde 2015 hasta la actualidad.
- Descarga: por año, mes y tipo de contratación, en formato JSON, CSV o Excel. Para este proyecto se descargó **JSON, año por año, con tipo de contratación = "Todos los tipos"**, porque conserva la estructura anidada completa del estándar OCDS (releases, comprador, oferta, adjudicaciones, contratos), necesaria para construir los indicadores de riesgo con el mayor detalle posible.
- Se trabajó con 12 archivos anuales (`releases_2015.json` ... `releases_2026.json`), descargados manualmente desde el portal (el sitio bloquea el acceso automatizado/scraping).

## 3. ¿Por qué DuckDB y por qué Parquet?

**El problema**: los archivos JSON anuales de SERCOP pesan entre 1 y 2 GB cada uno, y cada "paquete" contiene un objeto anidado con varios niveles (comprador, oferta, adjudicaciones, contratos, partes). Cargar esto directamente con `pandas.read_json` sería lento y consumiría mucha memoria, porque pandas necesita materializar todo el árbol de objetos en Python antes de poder trabajarlo como tabla.

**DuckDB** es una base de datos analítica que se ejecuta embebida en Python (como SQLite, pero optimizada para análisis de datos, no para transacciones). Se eligió porque:

- Puede leer JSON anidado **directamente** con `read_json_auto`, infiriendo el esquema automáticamente y permitiendo acceder a campos anidados con notación de punto (`r.buyer.name`, `r.tender.value.amount`) sin tener que aplanarlo manualmente en Python primero.
- Procesa los datos de forma eficiente en memoria, sin necesidad de cargar el archivo completo en RAM de una sola vez como haría pandas.
- Permite ejecutar SQL estándar directamente sobre archivos (JSON o Parquet) sin necesidad de un servidor de base de datos aparte.

**Parquet** es el formato en el que se guarda el resultado ya aplanado (una fila por proceso de contratación, con columnas planas). Se eligió porque:

- Es un formato **columnar**: al analizar solo algunas columnas (por ejemplo, monto y método de contratación), no hay que leer el archivo completo, lo que acelera mucho las consultas.
- Ocupa mucho menos espacio en disco que JSON o CSV, porque comprime los datos por columna.
- Es el formato "puente" natural entre DuckDB, pandas y scikit-learn — se puede leer y escribir desde cualquiera de los tres sin conversiones adicionales.

**Flujo general del pipeline**:

```
JSON anidado (por año, ~1-2 GB c/u)
      │
      ▼  DuckDB (read_json_auto + UNNEST)
Tabla plana en memoria
      │
      ▼  COPY ... TO archivo.parquet
Parquet por año
      │
      ▼  UNION ALL (DuckDB)
Parquet consolidado (2015-2026, ~2.78 millones de filas)
```

## 4. Entorno de trabajo

El procesamiento se realizó en **local** (Visual Studio Code + Python), no en Google Colab, debido a los límites de memoria y de subida de archivos que se encontraron al trabajar con los 12 años completos (~19 GB en total) en la nube. El equipo cuenta con una máquina con procesador i7, 32 GB de RAM y 1 TB de almacenamiento, suficiente para procesar los archivos completos sin fragmentarlos.

## 5. Instalación de librerías

In [ ]:
pip install duckdb pandas pyarrow jupyter ipykernel matplotlib seaborn scikit-learn

## 6. Verificación de los archivos descargados

Antes de procesar nada, se confirma que los 12 archivos anuales están presentes y se revisa su peso en disco — un primer chequeo simple, pero útil para detectar de entrada si alguna descarga quedó incompleta.

In [ ]:
import duckdb, os, glob

con = duckdb.connect()

carpeta = r'C:\Users\chivi\OneDrive\Desktop\released'

archivos = sorted(glob.glob(os.path.join(carpeta, 'releases_*.json')))
print(f'Archivos encontrados: {len(archivos)}')
for a in archivos:
    print(' -', os.path.basename(a), f'({os.path.getsize(a)/1_000_000:.1f} MB)')

**Resultado**: los 12 archivos están presentes, con un peso total aproximado de 19 GB (entre ~958 MB y ~2,091 MB cada uno).

## 7. EDA individual por año — antes de unir nada

Antes de construir la base consolidada, se hizo un análisis exploratorio **por separado, año por año**, para responder una pregunta clave: *¿los datos de cada archivo son consistentes y confiables antes de mezclarlos?* Mezclar primero y detectar problemas después habría significado rehacer el trabajo de unión repetidas veces.

Por cada año, el siguiente bloque calcula:

- **Cobertura de fechas**: para detectar si el archivo realmente cubre el año completo o si hay meses faltantes.
- **Distribución de etapas (`tag`)**: qué combinación de etapas (`planning`, `tender`, `award`, `contract`, `implementation`) alcanzó cada proceso.
- **Método de contratación** (`procurementMethod`): `direct`, `selective`, `open`, `limited`.
- **Calidad del monto**: cuántos valores nulos o en cero hay, y el rango de valores.

Se usa `read_json_auto(..., format='array')` para leer el arreglo JSON, y `UNNEST` para "desempacar" el array `releases` de cada paquete en filas individuales — una fila por release, que es la unidad de análisis del proyecto.

In [ ]:
import duckdb, os, glob
import pandas as pd

con = duckdb.connect()
con.execute("SET memory_limit='24GB';")   # deja algo de margen sobre tus 32GB para el sistema

carpeta = r'C:\Users\chivi\OneDrive\Desktop\released'
archivos = sorted(glob.glob(os.path.join(carpeta, 'releases_*.json')))

resumen = []

for archivo in archivos:
    anio = os.path.basename(archivo).replace('releases_', '').replace('.json', '')
    print(f'\n{"="*60}')
    print(f'AÑO: {anio}')
    print(f'{"="*60}')

    try:
        n_paquetes = con.sql(f"""
            SELECT count(*) FROM read_json_auto('{archivo}', format='array')
        """).fetchone()[0]

        n_releases = con.sql(f"""
            SELECT count(*)
            FROM read_json_auto('{archivo}', format='array') AS t,
            UNNEST(t.releases) AS u(r)
        """).fetchone()[0]

        fechas = con.sql(f"""
            SELECT min(r.date) AS desde, max(r.date) AS hasta
            FROM read_json_auto('{archivo}', format='array') AS t,
            UNNEST(t.releases) AS u(r)
        """).fetchone()

        por_mes = con.sql(f"""
            SELECT strftime(CAST(r.date AS TIMESTAMP), '%Y-%m') AS mes, count(*) n
            FROM read_json_auto('{archivo}', format='array') AS t,
            UNNEST(t.releases) AS u(r)
            GROUP BY 1 ORDER BY 1
        """).df()

        tags = con.sql(f"""
            SELECT r.tag AS etapas, count(*) n
            FROM read_json_auto('{archivo}', format='array') AS t,
            UNNEST(t.releases) AS u(r)
            GROUP BY 1 ORDER BY n DESC
        """).df()

        metodos = con.sql(f"""
            SELECT r.tender.procurementMethod AS metodo, count(*) n
            FROM read_json_auto('{archivo}', format='array') AS t,
            UNNEST(t.releases) AS u(r)
            GROUP BY 1 ORDER BY n DESC
        """).df()

        montos = con.sql(f"""
            SELECT
                count(*) AS total,
                sum(CASE WHEN r.tender.value.amount IS NULL THEN 1 ELSE 0 END) AS monto_nulo,
                sum(CASE WHEN r.tender.value.amount = 0 THEN 1 ELSE 0 END) AS monto_cero,
                min(r.tender.value.amount) AS min_monto,
                max(r.tender.value.amount) AS max_monto,
                avg(r.tender.value.amount) AS promedio_monto
            FROM read_json_auto('{archivo}', format='array') AS t,
            UNNEST(t.releases) AS u(r)
        """).df()

        print(f'Paquetes: {n_paquetes} | Releases: {n_releases} | Meses cubiertos: {len(por_mes)}')
        print(f'Rango de fechas: {fechas[0]} → {fechas[1]}')
        print('Etapas (tag):')
        print(tags.to_string(index=False))
        print('Método de contratación:')
        print(metodos.to_string(index=False))
        print('Calidad del monto:')
        print(montos.to_string(index=False))

        resumen.append({
            'anio': anio, 'n_paquetes': n_paquetes, 'n_releases': n_releases,
            'desde': fechas[0], 'hasta': fechas[1], 'meses_cubiertos': len(por_mes)
        })

    except Exception as e:
        print(f'⚠️ ERROR en {anio}: {e}')
        resumen.append({'anio': anio, 'ERROR': str(e)})

print('\n\n' + '='*60)
print('RESUMEN COMPARATIVO — TODOS LOS AÑOS')
print('='*60)
print(pd.DataFrame(resumen).to_string(index=False))

### Hallazgos de este EDA por año

**1. El archivo de 2024 falló al leerse** con el error:

```
Invalid Input Error: JSON transform error [...] record/value 63374: Expected ARRAY, but got OBJECT
```

Este error se investiga y se resuelve en la **Parte 3**.

**2. El campo `date` "se derrama" a años posteriores desde 2020.** Los archivos de 2020 en adelante muestran un rango de fechas que se extiende hasta 1-2 años después del año nominal del archivo (por ejemplo, el archivo `releases_2020.json` contiene releases con fecha hasta `2021-12-31`). Esto se debe a que `date` representa la fecha de la **última actualización** de esa versión del release, no la fecha de inicio del proceso — un proceso que arrancó en 2020 puede seguir actualizándose (por ejemplo, llegando a la etapa `implementation`) años después, y esa actualización queda registrada con una fecha posterior, aunque el proceso completo se descargó como parte del archivo de 2020.

Esto generó una duda importante: **¿esto significa que el mismo proceso puede aparecer duplicado entre el archivo de un año y el del año siguiente?** Esta pregunta se responde empíricamente en la Parte 3, antes de unir los archivos.

**3. Método de contratación y calidad del monto son consistentes año a año**, con predominancia de `selective` y `direct` sobre `open`, y una proporción de montos nulos que ronda el 5-10% en todos los años — no se identificó ningún año con una caída anómala de calidad respecto a los demás (aparte del error de lectura de 2024).

## 8. Investigación del error de lectura en el archivo 2024

El error reportado por DuckDB fue:

```
Invalid Input Error: JSON transform error [...] record/value 63374: Expected ARRAY, but got OBJECT: {"0":{"id":"2266965-COTB-VIALSUREP-2024-002",...
```

Esto indica que en el paquete de la posición 63,374 del archivo, un campo que en la mayoría de los procesos es una **lista** (`[...]`) viene serializado como un **objeto con claves numéricas de texto** (`{"0": {...}, "1": {...}}`). Es un patrón típico de sistemas backend construidos en PHP: cuando un array de PHP pierde la secuencia de índices (por ejemplo, se elimina un elemento intermedio), PHP lo serializa como objeto JSON en vez de arreglo. Antes de decidir cómo corregirlo, se inspecciona el registro exacto para confirmar en qué campo ocurre.

In [ ]:
import json

archivo_2024 = r'C:\Users\chivi\OneDrive\Desktop\released\releases_2024.json'

with open(archivo_2024, encoding='utf-8') as f:
    data = json.load(f)

print(f'Total de paquetes: {len(data)}')

registro_malo = data[63374]
print(json.dumps(registro_malo, indent=2, ensure_ascii=False)[:3000])

Con el registro a la vista, se buscó de forma sistemática (no solo en ese caso puntual) si había otros campos con el mismo patrón, recorriendo recursivamente cada paquete. La función `buscar_arrays_como_objeto` detecta cualquier diccionario cuyas claves sean **todas dígitos** (`"0"`, `"1"`, `"2"`, ...), que es la firma de un array mal serializado como objeto.

**Primer intento**: por razones de tiempo, se probó primero solo sobre los primeros 5,000 registros del archivo — pero el registro problemático (63,374) queda fuera de esa muestra, así que el resultado salió vacío.

In [ ]:
def buscar_arrays_como_objeto(obj, ruta=''):
    """Busca recursivamente objetos con forma {"0": ..., "1": ...} que deberían ser arrays."""
    hallazgos = []
    if isinstance(obj, dict):
        claves = list(obj.keys())
        # Si TODAS las claves son dígitos consecutivos empezando en 0, es un array disfrazado
        if claves and all(k.isdigit() for k in claves):
            hallazgos.append(ruta)
        for k, v in obj.items():
            hallazgos += buscar_arrays_como_objeto(v, f'{ruta}.{k}')
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            hallazgos += buscar_arrays_como_objeto(v, f'{ruta}[{i}]')
    return hallazgos

from collections import Counter
import re

patrones = Counter()
for i, paquete in enumerate(data[:5000]):
    hallazgos = buscar_arrays_como_objeto(paquete)
    for h in hallazgos:
        h_norm = re.sub(r'\[\d+\]', '[N]', h)
        patrones[h_norm] += 1

print('Campos afectados y cuántas veces aparecen (en los primeros 5000 registros):')
for campo, n in patrones.most_common():
    print(f'  {campo}: {n}')

**Resultado del primer intento**: ningún hallazgo (el conteo salió vacío), confirmando que el registro problemático está fuera del rango de muestra revisado.

**Segundo intento**: se repite la búsqueda acotando el rango a `data[60000:65000]`, que sí incluye la posición 63,374 donde ocurrió el error original.

In [ ]:
import json

# Si ya tienes 'data' cargado de antes, puedes saltarte este bloque
archivo_2024 = r'C:\Users\chivi\OneDrive\Desktop\released\releases_2024.json'
with open(archivo_2024, encoding='utf-8') as f:
    data = json.load(f)

print(f'Total de paquetes: {len(data)}')

In [ ]:
from collections import Counter
import re

def buscar_arrays_como_objeto(obj, ruta=''):
    """Busca recursivamente objetos con forma {"0": ..., "1": ...} que deberían ser arrays."""
    hallazgos = []
    if isinstance(obj, dict):
        claves = list(obj.keys())
        if claves and all(k.isdigit() for k in claves):
            hallazgos.append(ruta)
        for k, v in obj.items():
            hallazgos += buscar_arrays_como_objeto(v, f'{ruta}.{k}')
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            hallazgos += buscar_arrays_como_objeto(v, f'{ruta}[{i}]')
    return hallazgos

# Esta vez sí incluye la posición 63374
rango_inicio, rango_fin = 60000, 65000
patrones = Counter()
posiciones_afectadas = []

for i in range(rango_inicio, min(rango_fin, len(data))):
    hallazgos = buscar_arrays_como_objeto(data[i])
    if hallazgos:
        posiciones_afectadas.append(i)
    for h in hallazgos:
        h_norm = re.sub(r'\[\d+\]', '[N]', h)
        patrones[h_norm] += 1

print(f'Registros afectados en el rango {rango_inicio}-{rango_fin}: {len(posiciones_afectadas)}')
print('Primeras posiciones afectadas:', posiciones_afectadas[:10])
print('\nCampos afectados y cuántas veces aparecen:')
for campo, n in patrones.most_common():
    print(f'  {campo}: {n}')

**Resultado**: 1 solo registro afectado en el rango revisado (posición 61,595), y el campo exacto es `.releases[N].contracts` — es decir, el campo `contracts` de un release específico vino como objeto en vez de lista.

Para confirmar el alcance real del problema (no solo en la muestra, sino en **todo** el archivo), se recorrió el archivo completo contando cuántos releases tienen `contracts` como diccionario.

In [ ]:
afectados = []
for i, paquete in enumerate(data):
    for j, release in enumerate(paquete.get('releases', [])):
        contratos = release.get('contracts')
        if isinstance(contratos, dict):
            afectados.append((i, j))

print(f'Total de releases con "contracts" mal serializado: {len(afectados)}')
print(f'Total de releases en el archivo: {sum(len(p["releases"]) for p in data)}')
print('Primeras posiciones:', afectados[:10])

> **Nota de calidad adicional (sin resolver)**: la segunda línea de este bloque (`sum(len(p["releases"]) for p in data)`) lanzó un `KeyError: 'releases'` al ejecutarse, después de imprimir correctamente el primer resultado (`Total de releases con "contracts" mal serializado: 1`). Esto indica que **al menos un paquete del archivo de 2024 no tiene la clave `releases`**, un segundo problema de estructura distinto al de `contracts`. No se investigó a fondo cuál paquete es ni por qué le falta esa clave, porque no bloqueó el resto del pipeline (la reparación de `contracts` en el siguiente paso no depende de esta línea). Queda documentado como un hallazgo de calidad adicional pendiente de investigación más detallada — por ejemplo, revisando cuántos paquetes en total carecen de `releases` y si eso representa una pérdida real de procesos en la base consolidada.

## 9. Reparación del campo `contracts`

Con el alcance confirmado (1 registro afectado de 219,186 paquetes, es decir 0.0005% del archivo), se optó por **reparar** el campo en vez de descartar el proceso completo, ya que es un solo caso y el resto del registro es información real y utilizable. La función `reparar_contracts` detecta el patrón de objeto-con-claves-numéricas y lo reconstruye como lista, respetando el orden original de las claves.

In [ ]:
def reparar_contracts(release):
    """Si 'contracts' vino como objeto {'0':..,'1':..} en vez de lista, lo convierte a lista."""
    contratos = release.get('contracts')
    if isinstance(contratos, dict):
        # Ordenamos por la clave numérica para no perder el orden original
        claves_ordenadas = sorted(contratos.keys(), key=lambda k: int(k))
        release['contracts'] = [contratos[k] for k in claves_ordenadas]
    return release

# Aplicamos la reparación a todo el archivo
n_reparados = 0
for paquete in data:
    for release in paquete.get('releases', []):
        antes = isinstance(release.get('contracts'), dict)
        reparar_contracts(release)
        if antes:
            n_reparados += 1

print(f'Registros reparados: {n_reparados}')

# Guardamos el archivo ya corregido, con otro nombre para no perder el original
archivo_reparado = r'C:\Users\chivi\OneDrive\Desktop\released\releases_2024_reparado.json'
with open(archivo_reparado, 'w', encoding='utf-8') as f:
    json.dump(data, f, ensure_ascii=False)

print(f'Guardado en: {archivo_reparado}')

Se conserva el archivo original sin modificar y se guarda la versión reparada con otro nombre (`releases_2024_reparado.json`), para mantener trazabilidad de la corrección. Se verifica que el archivo reparado ahora sí se puede leer completo con DuckDB:

In [ ]:
import duckdb
con = duckdb.connect()

n_releases = con.sql(f"""
    SELECT count(*)
    FROM read_json_auto('{archivo_reparado}', format='array') AS t,
    UNNEST(t.releases) AS u(r)
""").fetchone()[0]

print(f'Releases leídos correctamente: {n_releases}')

**Resultado**: 219,185 releases leídos correctamente (sobre 219,186 paquetes) — el ligero desfase de 1 es consistente con el hallazgo de la Nota anterior (probablemente el paquete sin `releases`, que no aporta ninguna fila al hacer `UNNEST`).

## 10. ¿Hay procesos duplicados entre archivos de años consecutivos?

Antes de unir los 12 años en una sola base, había que resolver la duda que quedó pendiente de la Parte 2: si el campo `date` de un proceso de 2020 puede tener fecha de 2021, **¿ese mismo proceso podría aparecer también en el archivo `releases_2021.json`, duplicado?**

En vez de asumir una respuesta, se verificó empíricamente: se extrajeron todos los `ocid` únicos de cada uno de los 12 archivos (usando ya el archivo reparado para 2024) y se comparó la intersección entre cada par de años consecutivos.

In [ ]:
import duckdb, os

con = duckdb.connect()
con.execute("SET memory_limit='24GB';")

carpeta = r'C:\Users\chivi\OneDrive\Desktop\released'

# Usa el archivo reparado para 2024, los originales para el resto
archivos = {
    '2015': 'releases_2015.json',
    '2016': 'releases_2016.json',
    '2017': 'releases_2017.json',
    '2018': 'releases_2018.json',
    '2019': 'releases_2019.json',
    '2020': 'releases_2020.json',
    '2021': 'releases_2021.json',
    '2022': 'releases_2022.json',
    '2023': 'releases_2023.json',
    '2024': 'releases_2024_reparado.json',
    '2025': 'releases_2025.json',
    '2026': 'releases_2026.json',
}

ocids_por_anio = {}

for anio, nombre in archivos.items():
    ruta = os.path.join(carpeta, nombre)
    ocids = con.sql(f"""
        SELECT DISTINCT r.ocid
        FROM read_json_auto('{ruta}', format='array') AS t,
        UNNEST(t.releases) AS u(r)
    """).df()['ocid']
    ocids_por_anio[anio] = set(ocids)
    print(f'{anio}: {len(ocids_por_anio[anio])} ocids únicos')

In [ ]:
anios_ordenados = sorted(ocids_por_anio.keys())

print('Solapamiento entre años consecutivos:')
for i in range(len(anios_ordenados) - 1):
    a1, a2 = anios_ordenados[i], anios_ordenados[i + 1]
    interseccion = ocids_por_anio[a1] & ocids_por_anio[a2]
    print(f'  {a1} ∩ {a2}: {len(interseccion)} ocids repetidos')

# Conteo global: cuántos ocids aparecen en más de un archivo anual
from collections import Counter
contador = Counter()
for ocids in ocids_por_anio.values():
    contador.update(ocids)

duplicados = {o: c for o, c in contador.items() if c > 1}
print(f'\nTotal de ocids que aparecen en más de un archivo anual: {len(duplicados)}')
if duplicados:
    ejemplo = list(duplicados.items())[0]
    print('Ejemplo:', ejemplo)

**Resultado: 0 ocids duplicados** entre todos los pares de años consecutivos, y 0 en total considerando los 12 archivos completos. Esto confirma que el portal SERCOP filtra la descarga anual por el **año de origen del proceso** (probablemente embebido en el propio `ocid`), no por la fecha de la última actualización — el "derrame" de fechas hacia años posteriores es real, pero cada proceso permanece íntegro dentro de un único archivo anual. Con esto confirmado, se puede unir los 12 archivos con una simple concatenación (`UNION ALL`), sin necesidad de lógica de deduplicación.

## 11. Unión de los 12 años en una base Parquet consolidada

Se construye una consulta `SELECT` por cada año (aplanando los mismos campos usados en el EDA de la Parte 2, más algunos adicionales: comprador, tipo y método de contratación, monto, oferentes, fechas de oferta, y conteos de adjudicaciones/partes/contratos), y se combinan con `UNION ALL BY NAME` — la variante de `UNION ALL` que empareja columnas por nombre en vez de por posición, más segura si algún año tuviera columnas en distinto orden. Se agrega una columna `anio_archivo` para poder rastrear de qué descarga original vino cada fila. El resultado se guarda directamente en un único archivo Parquet.

In [ ]:
import duckdb, os

con = duckdb.connect()
con.execute("SET memory_limit='24GB';")

carpeta = r'C:\Users\chivi\OneDrive\Desktop\released'
salida = r'C:\Users\chivi\OneDrive\Desktop\released\sercop_consolidado.parquet'

# Usamos el archivo reparado para 2024, los originales para el resto
archivos = {
    '2015': 'releases_2015.json',
    '2016': 'releases_2016.json',
    '2017': 'releases_2017.json',
    '2018': 'releases_2018.json',
    '2019': 'releases_2019.json',
    '2020': 'releases_2020.json',
    '2021': 'releases_2021.json',
    '2022': 'releases_2022.json',
    '2023': 'releases_2023.json',
    '2024': 'releases_2024_reparado.json',
    '2025': 'releases_2025.json',
    '2026': 'releases_2026.json',
}

# Construimos un SELECT por año y los pegamos con UNION ALL,
# agregando una columna 'anio_archivo' para saber de qué descarga vino cada fila
selects = []
for anio, nombre in archivos.items():
    ruta = os.path.join(carpeta, nombre).replace('\\', '/')
    selects.append(f"""
        SELECT
            '{anio}' AS anio_archivo,
            r.ocid AS ocid,
            r.date AS fecha_release,
            r.tag AS etapas,
            r.buyer.id AS buyer_id,
            r.buyer.name AS buyer_name,
            r.tender.procurementMethod AS metodo,
            r.tender.procurementMethodDetails AS tipo_contratacion,
            r.tender.value.amount AS monto_tender,
            r.tender.numberOfTenderers AS n_oferentes,
            r.tender.tenderPeriod.startDate AS fecha_inicio_tender,
            r.tender.tenderPeriod.endDate AS fecha_fin_tender,
            len(r.awards) AS n_awards,
            len(r.parties) AS n_parties,
            len(r.contracts) AS n_contracts
        FROM read_json_auto('{ruta}', format='array', union_by_name=true) AS t,
        UNNEST(t.releases) AS u(r)
    """)

query_final = " UNION ALL BY NAME ".join(selects)

con.sql(f"""
    COPY ({query_final}) TO '{salida.replace(chr(92), '/')}' (FORMAT PARQUET)
""")

print('✅ Base consolidada guardada en:', salida)

In [ ]:
total = con.sql(f"SELECT count(*) FROM '{salida.replace(chr(92), '/')}'").fetchone()[0]
print(f'Total de filas en la base consolidada: {total}')

por_anio = con.sql(f"""
    SELECT anio_archivo, count(*) n
    FROM '{salida.replace(chr(92), '/')}'
    GROUP BY 1 ORDER BY 1
""").df()
print(por_anio.to_string(index=False))

**Resultado: 2,785,355 filas totales**, coincidiendo exactamente con la suma de los 12 conteos individuales por año obtenidos en el EDA de la Parte 2 — confirmando que la unión no perdió ni duplicó ningún registro.

| Año | Filas |
|---|---|
| 2015 | 280,643 |
| 2016 | 279,636 |
| 2017 | 334,275 |
| 2018 | 357,687 |
| 2019 | 275,055 |
| 2020 | 160,676 |
| 2021 | 167,058 |
| 2022 | 212,324 |
| 2023 | 217,913 |
| 2024 | 219,185 |
| 2025 | 189,285 |
| 2026 | 91,618 |

Con esto queda construida la **base de datos consolidada 2015-2026**, punto de partida para el análisis exploratorio general (Parte 4) y la construcción de los indicadores de riesgo (Parte 6).